# Human-Centered Personas at Scale

**Using LLMs and RAG to Build Personas from Mixed-Methods Data**

SIOP 2026 Master Tutorial | Chelsea Wymer, Diana Wolfe, Alice Choe

---

## What You Are About to Do

You are going to take raw employee survey data and turn it into empirically grounded workforce personas. Along the way, you will:

1. **Screen** the data for quality problems using published best practices
2. **Discover** workforce segments using two independent clustering methods that challenge each other
3. **Ground** those segments in real organizational documents so the personas reflect actual policy, not just numbers
4. **Write** evidence-based persona narratives where every claim traces back to data

At each phase transition, you will stop at a **decision gate** where you -- the I-O psychologist -- make the call. The AI does the computation. You make the judgments.

### The Pipeline

![Pipeline Architecture](../resources/pipeline_diagram.png)

### How This Notebook Works

Run cells in order. Each phase explains the science before you run any code, so you understand *why* before you see *what*. Gate cells will ask you questions and record your decisions into an audit trail. Every output is saved as a human-readable report (.md), a reviewable spreadsheet (.csv), and a raw JSON file.

Every agent in this pipeline makes live calls to the Anthropic API (`ANTHROPIC_API_KEY` must be set in your environment). Statistical computation runs locally; the LLM provides expert reasoning and interpretation at each phase.

### The Pipeline at a Glance

| Phase | Agents | What Happens | Your Decision |
|-------|--------|--------------|---------------|
| 1. Ingest & Clean | Data Steward | Multi-hurdle quality screening | Accept or adjust screening criteria |
| 2. Discover Segments | K-Prototypes + LPA + Psychometrician | Two clustering methods run in parallel, then validated | Accept cluster solution |
| 3. Ground in Reality | RAG + Emergence | Retrieve org documents, detect novel themes | Review emergent themes |
| 4. Write Personas | Narrator | Generate evidence-grounded narratives | Approve personas for leadership |
| Bonus: Longitudinal | Continuity + Emergence | Map follow-up respondents to baseline groups | Review migration patterns |

---

## Setup

Run the cell below to load all dependencies and check your environment. If any import fails, run `pip install -r requirements.txt` from the repo root.

In [ ]:
import sys, os, uuid
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime, timezone
from IPython.display import display, Markdown, HTML

from src import config
from src.utils import (
    display_table, display_report, save_output, save_json,
    build_report, save_phase_outputs, save_audit_csv,
    write_success_report,
)
from src.project_manager import (
    create_audit_trail, add_entry, check_governance, generate_audit_report,
)
from src.reporting import (
    render_phase1_report, render_phase2_report,
    render_phase3_report, render_phase4_overview, render_persona_card,
)

# Phase imports -- notice how each maps to a phase in the diagram
from src.p1_ingest import run_data_steward
from src.p2_discover import run_k_prototypes, run_lpa, run_validation
from src.p3_ground import (
    build_knowledge_base, query_knowledge_base,
    ground_constructs, detect_emergent_themes,
)
from src.p4_narrate import generate_personas
from src.p5_longitudinal import align_to_baseline, test_new_segments

# Configure plotting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

# Run ID for this pipeline execution
RUN_ID = f"run_{datetime.now(timezone.utc).strftime('%Y_%m_%d_%H%M')}"

print(f"Model:  {config.MODEL}")
print(f"Seed:   {config.SEED}")
print(f"Run ID: {RUN_ID}")
print(f"\nAll imports loaded successfully.")
print("An ANTHROPIC_API_KEY environment variable is required to run this notebook.")

In [2]:
# Initialize the audit trail -- this follows you through every phase
audit_trail = create_audit_trail()
print("Audit trail initialized. Every agent action and gate decision will be logged.")

Audit trail initialized. Every agent action and gate decision will be logged.


---

# Phase 1: Ingest and Clean

## The Science

Before any analysis, survey data must be screened for quality. This is not optional housekeeping -- careless responding and low-quality items can distort cluster solutions and produce personas that describe noise rather than real workforce segments.

The Data Steward implements the **Survey Data Quality Evaluation Model** (Papp et al., 2026) with five sequential gates:

1. **Schema validation** -- Are the expected columns present? Are types correct?
2. **Careless responding detection** -- A multi-hurdle approach (Curran, 2016) that requires respondents to fail 2+ independent indicators before removal. This avoids false positives from any single metric.
   - *Longstring analysis*: Flags respondents who gave the same answer for long consecutive runs (straight-lining)
   - *Intra-individual response variability (IRV)*: Flags respondents with suspiciously low variance across items
3. **Sparsity gate** -- Columns with >20% missing values are flagged. Remaining missing values are imputed (median for numeric, "Unknown" for categorical).
4. **Variance gate** -- Survey items with SD < 0.5 are excluded. If everyone answered the same way, the item cannot distinguish between groups.
5. **Distribution screening** -- Checks skewness, kurtosis, and IQR outlier rates.

**Design decision:** The Data Steward does *not* standardize scores. That is deliberately delegated to downstream agents (K-Prototypes and LPA), because each algorithm may need different scaling. This prevents double-standardization errors.

## The Data

The synthetic dataset represents **Meridian Technologies**, a mid-size company undergoing a major reorganization. The survey measures five constructs on Likert scales:

| Column | Construct | What It Measures |
|--------|-----------|------------------|
| `Cared_About` | Perceived Organizational Support | "I feel the organization cares about me" |
| `Excited` | Work Engagement | "I am excited about the work I do" |
| `Helpful_Info` | Communication Effectiveness | "I receive helpful information about changes" |
| `Trust_Leadership` | Trust in Leadership | "I trust senior leadership's decisions" |
| `Morale` | Work Engagement (affective) | "Overall morale in my team is good" |

Plus four demographic columns: `Business Unit`, `Level`, `FLSA`, `Tenure`. The raw data also includes `Attention check 1`, `Attention check 2`, and `Comments` columns, which the Data Steward uses for careless responding detection before dropping them.

**Survey data files:**
- `synthetic_data/survey_baseline.csv` — baseline wave (used for all phases)
- `synthetic_data/survey_followup.csv` — follow-up wave (Bonus: Longitudinal phase only)

In [ ]:
# Load the baseline and follow-up surveys
baseline_df = pd.read_csv(os.path.join("..", "synthetic_data", "survey_baseline.csv"))

print(f"Baseline survey: {baseline_df.shape[0]:,} respondents, {baseline_df.shape[1]} columns")

# Load follow-up survey if it exists
try:
    followup_df = pd.read_csv(os.path.join("..", "synthetic_data", "survey_followup.csv"))
    print(f"Follow-up survey: {followup_df.shape[0]:,} respondents, {followup_df.shape[1]} columns")
except FileNotFoundError:
    followup_df = None
    print("Follow-up survey not found. Skipping longitudinal analysis.")

# Display baseline survey
print(f"\n--- First 10 rows ---")
display(baseline_df.head(10))
print(f"\n--- Descriptive statistics ---")
display(baseline_df.describe().round(3))

### Run the Data Steward

The cell below runs the full screening pipeline. Watch the reasoning output -- it explains every decision the agent made and why.

In [ ]:
steward_result = run_data_steward(baseline_df)

clean_df = steward_result["clean_df"]
quality_report = steward_result["quality_report"]
audit_trail.extend(steward_result["audit_entries"])

print("=" * 70)
print("DATA STEWARD REPORT")
print("=" * 70)
print(f"\nOriginal respondents:  {quality_report['original_rows']:,}")
print(f"Removed (careless):    {steward_result['removed_count']}")
print(f"Clean respondents:     {quality_report['clean_rows']:,}")
print(f"Survey items retained: {len(quality_report['retained_survey_cols'])}")
print(f"Data quality score:    {quality_report['confidence']:.2%}")
print(f"\n--- Agent Reasoning ---")
print(steward_result["reasoning"])

In [ ]:
# ── Phase 1 rich output ──────────────────────────────────────────────────
display(HTML(render_phase1_report(steward_result, quality_report,
                                  baseline_df=baseline_df, run_id=RUN_ID)))

In [ ]:
# Visualize the screening results
cr = quality_report["careless_responding"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Careless responding breakdown
indicators = ["Longstring\nflagged", "IRV\nflagged", "Multi-hurdle\nremoved"]
counts = [cr["longstring_flagged"], cr["irv_flagged"], cr["multi_hurdle_flagged"]]
colors = ["#5b9bd5", "#5b9bd5", "#c0392b"]
ax1.bar(indicators, counts, color=colors)
ax1.set_ylabel("Respondents")
ax1.set_title("Careless Responding Screening")
for i, v in enumerate(counts):
    ax1.text(i, v + max(counts) * 0.02, str(v), ha="center", fontweight="bold")

# Variance by item
var_stats = quality_report["variance"]["stats"]
items = list(var_stats.keys())
sds = [var_stats[c]["sd"] for c in items]
bar_colors = ["#c0392b" if c in quality_report["variance"]["excluded"] else "#5b9bd5" for c in items]
ax2.barh(items, sds, color=bar_colors)
ax2.axvline(x=config.VARIANCE_THRESHOLD_SD, color="black", linestyle="--",
            label=f"Threshold (SD = {config.VARIANCE_THRESHOLD_SD})")
ax2.set_xlabel("Standard Deviation")
ax2.set_title("Variance Gate: Item Discrimination")
ax2.legend()

plt.tight_layout()
plt.show()

print("\nRed bars = excluded (cannot discriminate between groups)")
print("Blue bars = retained for clustering")

---

## Gate 1: Your Decision -- Data Quality

Review the screening results above. Consider:

- **Removal rate**: Is it in the typical 2-8% range (Curran, 2016)? If it is very low, the screening may be too lenient. If it is very high, check whether data collection conditions were unusual.
- **Variance gate**: Are the right items being retained? An item excluded here will not appear in any cluster solution.
- **Demographics**: Were removals proportionate across business units, levels, and tenure bands? Disproportionate removal could introduce bias.

### Ethics Checkpoint
- Could the screening criteria systematically disadvantage any demographic group?
- Is the data sufficiently de-identified for this analysis?
- If real data: did respondents consent to this type of analysis?

In [ ]:
print("GATE 1: DATA QUALITY REVIEW")
print("-" * 40)
gate1_decision = input("Do you accept the screening results? (yes / no / investigate): ")
gate1_notes = input("Any notes for the audit trail? (or press Enter to skip): ")

add_entry(audit_trail, "Gate 1", "Human", "Data quality review", {
    "decision": gate1_decision,
    "notes": gate1_notes,
    "removal_rate": f"{steward_result['removed_count'] / quality_report['original_rows'] * 100:.1f}%",
    "items_retained": quality_report["retained_survey_cols"],
})
print(f"\nDecision recorded: {gate1_decision}")
print("Proceeding to Phase 2.")

In [ ]:
# ── Phase 1 output paths ─────────────────────────────────────────────────
_P1 = os.path.join("..", "outputs", "phase1_data_quality_report")
_P1_AUDIT = os.path.join(_P1, "audit_reports")
_P1_LOGS  = os.path.join(_P1, "reflection_logs")
for _d in (_P1, _P1_AUDIT, _P1_LOGS):
    os.makedirs(_d, exist_ok=True)

# ── Main report ───────────────────────────────────────────────────────────
p1_report = build_report("Phase 1: Data Quality Report", [
    {"heading": "Agent Reasoning", "body": steward_result["reasoning"]},
    {"heading": "Summary Statistics", "body": (
        f"- Original respondents: {quality_report['original_rows']:,}\n"
        f"- Removed (careless): {steward_result['removed_count']}\n"
        f"- Clean respondents: {quality_report['clean_rows']:,}\n"
        f"- Items retained: {', '.join(quality_report['retained_survey_cols'])}\n"
        f"- Confidence: {quality_report['confidence']:.2%}"
    )},
    {"heading": "Gate 1 Decision", "body": f"{gate1_decision}. {gate1_notes}"},
])

screening_df = baseline_df.copy()
screening_df["retained"] = screening_df.index.isin(clean_df.index)

written = save_phase_outputs(
    _P1,
    report_md=p1_report,
    dataframes={"screening_results": screening_df},
    raw_json={"quality_report": quality_report, "reasoning": steward_result["reasoning"]},
)

# ── Clean respondent dataset ──────────────────────────────────────────────
_p = os.path.join(_P1, "survey_baseline_clean.csv")
clean_df.to_csv(_p, index=False)
written.append(_p)

# ── Audit trail ───────────────────────────────────────────────────────────
_p = save_json(steward_result["audit_entries"], os.path.join(_P1_AUDIT, "audit_trail.json"))
written.append(_p)

# ── Detailed screening report ─────────────────────────────────────────────
_cr  = quality_report["careless_responding"]
_var = quality_report["variance"]
_dist = quality_report["distributions"]

_screening_lines = [
    "# Data Steward: Detailed Screening Report", "",
    "## Careless Responding Hurdles", "",
    "| Indicator | Threshold | Flagged |",
    "|-----------|-----------|---------|",
    f"| Longstring | >{_cr['longstring_threshold']:.0f} consecutive identical | {_cr['longstring_flagged']} |",
    f"| IRV (within-person SD) | <0.20 | {_cr['irv_flagged']} |",
    f"| **Multi-hurdle removed** | ≥{config.CARELESS_HURDLES} indicators | **{_cr['multi_hurdle_flagged']}** |",
    "", "## Variance Gate", "",
    "| Item | SD | Result |", "|------|----|--------|",
] + [
    f"| {c} | {s['sd']:.4f} | {'**EXCLUDED**' if c in _var['excluded'] else 'retained'} |"
    for c, s in _var["stats"].items()
] + [
    "", "## Distribution Screening", "",
    "| Item | Skewness | Kurtosis | IQR Outliers | Flags |",
    "|------|----------|----------|-------------|-------|",
] + [
    f"| {d['column']} | {d['skewness']:.3f} | {d['kurtosis']:.3f} | {d['iqr_outliers']} | {'; '.join(d['issues']) or 'none'} |"
    for d in _dist
]

_p = os.path.join(_P1_AUDIT, "data_quality_screening.md")
with open(_p, "w", encoding="utf-8") as _f:
    _f.write("\n".join(_screening_lines))
written.append(_p)

# ── Success report ────────────────────────────────────────────────────────
_removal_pct = steward_result["removed_count"] / quality_report["original_rows"] * 100
_p = write_success_report(
    os.path.join(_P1_LOGS, "data_steward_success_report.txt"),
    phase="Phase 1 -- Ingest and Clean",
    agents="Data Steward",
    status=f"COMPLETE -- Gate 1: {gate1_decision}",
    metrics={
        "Original respondents":  f"{quality_report['original_rows']:,}",
        "Clean respondents":     f"{quality_report['clean_rows']:,}",
        "Removal rate":          f"{_removal_pct:.1f}%",
        "Items retained":        ", ".join(quality_report["retained_survey_cols"]),
        "Items excluded":        ", ".join(quality_report["variance"]["excluded"]) or "none",
        "Confidence score":      f"{quality_report['confidence']:.2%}",
    },
    artifacts=[os.path.relpath(f, _P1) for f in written] + [
        "reflection_logs/data_steward_success_report.txt"
    ],
    notes=(
        "Multi-hurdle careless responding screen (Papp et al., 2026; Curran, 2016). "
        "Standardisation withheld -- delegated to K-Prototypes and LPA agents."
    ),
)
written.append(_p)

print("Phase 1 outputs saved:")
for f in written:
    print(f"  {f}")

---

# Phase 2: Discover Workforce Segments

## The Science

This phase runs **two independent clustering methods** on the same data and compares their results. This is deliberately adversarial -- if two very different algorithms find similar groups, you can be more confident those groups are real.

### Method 1: K-Prototypes (Huang, 1998)

**What it does:** Groups people using *both* demographics and survey scores simultaneously.

**Why this method:** Most clustering algorithms handle only numeric data. But organizational survey data is inherently mixed-type: categorical demographics (Business Unit, Level) and continuous Likert scores. K-Prototypes solves this by combining K-Means (Euclidean distance on numerics) with K-Modes (Hamming distance on categoricals) through a gamma-weighted cost function.

**Key parameters:**
- *Gamma*: Controls the balance between numeric and categorical influence. Set from the data (mean SD of standardized numerics).
- *K*: Number of clusters. Selected by comparing the elbow method (cost reduction) with silhouette scores (cluster separation).
- *Initialization*: Cao method for reproducibility.

**What it answers:** *Who are these people?* -- segments defined by the intersection of demographics and attitudes.

### Method 2: Latent Profile Analysis (LPA) via GMM

**What it does:** Groups people using *only* survey scores. Demographics are ignored entirely.

**Why this method:** LPA is a model-based approach grounded in statistical theory (Spurk et al., 2020). Unlike K-Prototypes, which uses arbitrary distance thresholds, LPA assumes the data is generated by a mixture of Gaussian distributions and uses maximum likelihood to estimate group membership.

**Key parameters:**
- *BIC*: Bayesian Information Criterion -- the primary model selection tool. Balances fit against complexity.
- *Entropy*: Classification certainty. 1.0 = everyone clearly belongs to one group. <0.60 = substantial overlap.
- *Posterior probability*: Each person gets a probability of belonging to each profile. Below 0.70 = "ambiguous."

**What it answers:** *What psychological profiles exist?* -- segments defined purely by attitude patterns.

### The Psychometrician: Cross-Validation

After both methods run, the Psychometrician agent computes:
- **Silhouette scores**: How well-separated are the clusters? (Rousseeuw, 1987)
- **Adjusted Rand Index (ARI)**: How much do the two methods agree? Interpreted through an inter-rater reliability framework (Hallgren, 2012) -- the two algorithms are treated as independent "raters" classifying respondents.
- **Outlier flags**: Respondents far from any cluster centroid who may represent unmodeled experiences.

**Interpretation guide for ARI:**
- \>0.65 = Strong agreement (demographics and psychology tell the same story)
- 0.30-0.65 = Moderate (partially overlapping, which is often the most informative finding)
- <0.30 = Weak (the two methods see very different structure)

In [ ]:
# --- K-Prototypes: Mixed-type clustering ---
print("Running K-Prototypes (this groups by demographics AND survey scores)...")
kproto_result = run_k_prototypes(clean_df)
audit_trail.extend(kproto_result["audit_entries"])

print(f"\nSelected K = {kproto_result['k_selected']}")
print(f"\nCluster sizes:")
for _, row in kproto_result["profiles"].iterrows():
    print(f"  Cluster {int(row['cluster'])}: n={int(row['n'])} ({row['pct']}%)")
print(f"\n--- Agent Reasoning ---")
print(kproto_result["reasoning"])

In [ ]:
# Visualize K-Prototypes results
elbow = kproto_result["elbow_data"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(elbow["k_values"], elbow["costs"], "o-", linewidth=2, color="#2c3e50")
ax1.axvline(x=kproto_result["k_selected"], color="#c0392b", linestyle="--",
            label=f"Selected K={kproto_result['k_selected']}")
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Cost")
ax1.set_title("Elbow Plot")
ax1.legend()

sil_k = sorted(elbow["silhouette_scores"].keys())
sil_v = [elbow["silhouette_scores"][k] for k in sil_k]
ax2.bar(sil_k, sil_v, color="#5b9bd5")
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette by K")

plt.suptitle("K-Prototypes: Model Selection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profile heatmap
profiles = kproto_result["profiles"]
num_data = profiles.set_index("cluster")[config.NUMERIC_COLS]

fig, ax = plt.subplots(figsize=(10, max(3, len(profiles) * 0.8)))
sns.heatmap(num_data.astype(float), annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("K-Prototypes: Cluster Profiles (Mean Z-Scores by Construct)")
ax.set_ylabel("Cluster")
plt.tight_layout()
plt.show()

print("How to read this: Green = high on the construct, Red = low.")
print("Each row is a cluster. Look for distinct color patterns across rows.")
print("If two rows look identical, those clusters may not be meaningfully different.")

In [ ]:
# --- Latent Profile Analysis: Survey-score-only clustering ---
print("Running LPA (this groups by survey scores ONLY, ignoring demographics)...")
lpa_result = run_lpa(clean_df)
audit_trail.extend(lpa_result["audit_entries"])

selected = lpa_result["fit_indices"]["selected"]
print(f"\nSelected: {selected['K']} profiles ({selected['cov_type']} covariance)")
print(f"BIC: {selected['BIC']:.1f}")
print(f"Entropy: {selected['entropy']:.3f}")
print(f"Ambiguous respondents: {lpa_result['ambiguous_count']}")
print(f"\n--- Agent Reasoning ---")
print(lpa_result["reasoning"])

In [ ]:
# LPA Fit Indices Table
print("Model Comparison (all models tested):")
fit_df = pd.DataFrame(lpa_result["fit_indices"]["all_models"])
display(fit_df)

print("\n" + "=" * 50)
print("PSYCHOLOGICAL FINGERPRINTS")
print("=" * 50)
print("\nEach profile gets a fingerprint: High / Moderate / Low on each construct.")
print("High = z-score > 0.5, Low = z-score < -0.5, Moderate = in between.\n")

for pid, fp in lpa_result["fingerprints"].items():
    print(f"  Profile {pid}: {fp['label']}")
    for dim, level in fp["dims"].items():
        marker = {"High": "+", "Low": "-", "Moderate": "~"}.get(level, "?")
        print(f"    [{marker}] {dim}: {level} (z = {fp['means'][dim]:.2f})")
    print()

In [ ]:
# --- Psychometrician: Validate both solutions ---
print("Running Psychometrician (cross-validating the two solutions)...")
validation_result = run_validation(
    clean_df,
    labels_kproto=kproto_result["labels"],
    labels_lpa=lpa_result["labels"],
)
audit_trail.extend(validation_result["audit_entries"])

print(f"\n--- Agent Reasoning ---")
print(validation_result["reasoning"])

In [ ]:
# ── Phase 2 rich output ──────────────────────────────────────────────────
display(HTML(render_phase2_report(kproto_result, lpa_result, validation_result,
                                  run_id=RUN_ID)))

In [ ]:
# Visualize validation
spc = validation_result["silhouette_per_cluster"]

fig, ax = plt.subplots(figsize=(8, 4))
clusters = sorted(spc.keys())
means = [spc[c]["mean"] for c in clusters]
colors = ["#27ae60" if m > 0.25 else "#e67e22" if m > 0 else "#c0392b" for m in means]
ax.bar([f"Cluster {c}" for c in clusters], means, color=colors)
ax.axhline(y=validation_result["silhouette_overall"], color="black", linestyle="--",
           label=f"Overall: {validation_result['silhouette_overall']:.3f}")
ax.axhline(y=0.25, color="gray", linestyle=":", alpha=0.5, label="Moderate threshold (0.25)")
ax.set_ylabel("Mean Silhouette Score")
ax.set_title("Cluster Quality: Silhouette Scores")
ax.legend()
plt.tight_layout()
plt.show()

print("Green = good separation, Orange = fair, Red = weak.")
if validation_result["ari"] is not None:
    print(f"\nCross-Method ARI: {validation_result['ari']:.3f}")
    print(f"Outliers flagged: {int(validation_result['outlier_flags'].sum())}")

---

## Gate 2: Your Decision -- Cluster Solution

You now have two independent views of the workforce structure. Consider:

- **K-Prototypes** found segments using demographics + attitudes. How many? Are they distinct?
- **LPA** found profiles using attitudes only. Do the fingerprints tell a clear story?
- **ARI**: Do the methods agree? Moderate agreement (0.30-0.65) is actually the most common and informative finding -- it means demographics and psychology overlap but are not redundant.
- **Outliers**: How many respondents do not fit any cluster well?

### Ethics Checkpoint
- Do any clusters map suspiciously onto a single demographic group? If Cluster 3 is 95% one Business Unit, that is a segment discovery, not a bias -- but it needs careful framing.
- Are outliers being fairly treated? They are real people with real experiences that your model does not capture.

In [ ]:
print("GATE 2: CLUSTER VALIDATION REVIEW")
print("-" * 40)
gate2_decision = input("Do you accept the cluster solution? (yes / no / investigate): ")
gate2_notes = input("Any notes? (or Enter to skip): ")

add_entry(audit_trail, "Gate 2", "Human", "Cluster validation review", {
    "decision": gate2_decision,
    "notes": gate2_notes,
    "kproto_k": kproto_result["k_selected"],
    "lpa_k": lpa_result["fit_indices"]["selected"]["K"],
    "ari": validation_result["ari"],
    "silhouette": validation_result["silhouette_overall"],
})
print(f"\nDecision recorded: {gate2_decision}")

In [ ]:
# ── Phase 2 output paths ─────────────────────────────────────────────────
_P2 = os.path.join("..", "outputs", "phase2_cluster_validation")
_P2_AUDIT = os.path.join(_P2, "audit_reports")
_P2_LOGS  = os.path.join(_P2, "reflection_logs")
for _d in (_P2, _P2_AUDIT, _P2_LOGS):
    os.makedirs(_d, exist_ok=True)

# ── Main report ───────────────────────────────────────────────────────────
p2_report = build_report("Phase 2: Cluster Validation Summary", [
    {"heading": "K-Prototypes", "body": kproto_result["reasoning"]},
    {"heading": "Latent Profile Analysis", "body": lpa_result["reasoning"]},
    {"heading": "Psychometrician Validation", "body": validation_result["reasoning"]},
    {"heading": "Gate 2 Decision", "body": f"{gate2_decision}. {gate2_notes}"},
])

assignments_df = clean_df.copy()
assignments_df["KProto_Cluster"] = kproto_result["labels"]
assignments_df["LPA_Profile"]    = lpa_result["labels"]
assignments_df["Outlier"]        = validation_result["outlier_flags"]

written = save_phase_outputs(
    _P2,
    report_md=p2_report,
    dataframes={"cluster_assignments": assignments_df},
    raw_json={
        "kproto":      {"k": kproto_result["k_selected"], "reasoning": kproto_result["reasoning"]},
        "lpa":         {"k": lpa_result["fit_indices"]["selected"]["K"], "reasoning": lpa_result["reasoning"]},
        "validation":  {"silhouette": validation_result["silhouette_overall"],
                        "ari": validation_result["ari"], "reasoning": validation_result["reasoning"]},
    },
)

# ── K-Prototypes profiles + centroids ─────────────────────────────────────
_p = os.path.join(_P2, "kproto_profiles.csv")
kproto_result["profiles"].to_csv(_p, index=False)
written.append(_p)

_p = save_json(kproto_result["centroids"], os.path.join(_P2, "kproto_centroids.json"))
written.append(_p)

# ── LPA profiles + fingerprints ────────────────────────────────────────────
_p = os.path.join(_P2, "lpa_profiles.csv")
lpa_result["profiles"].to_csv(_p, index=False)
written.append(_p)

_p = save_json(lpa_result["fingerprints"], os.path.join(_P2, "lpa_fingerprints.json"))
written.append(_p)

# ── Validation summary JSON ────────────────────────────────────────────────
_p = save_json({
    "silhouette_overall":    validation_result["silhouette_overall"],
    "silhouette_per_cluster": validation_result["silhouette_per_cluster"],
    "ari":                   validation_result["ari"],
    "ari_interpretation":    validation_result["ari_interpretation"],
    "n_outliers":            int(validation_result["outlier_flags"].sum()),
}, os.path.join(_P2, "validation_summary.json"))
written.append(_p)

# ── Audit trail ───────────────────────────────────────────────────────────
_combined_audit = (
    kproto_result["audit_entries"]
    + lpa_result["audit_entries"]
    + validation_result["audit_entries"]
)
_p = save_json(_combined_audit, os.path.join(_P2_AUDIT, "audit_trail.json"))
written.append(_p)

# ── Success report ────────────────────────────────────────────────────────
_sil = validation_result["silhouette_overall"]
_ari = validation_result["ari"]
_p = write_success_report(
    os.path.join(_P2_LOGS, "phase2_success_report.txt"),
    phase="Phase 2 -- Discover Workforce Segments",
    agents=["K-Prototypes", "LPA", "Psychometrician"],
    status=f"COMPLETE -- Gate 2: {gate2_decision}",
    metrics={
        "Respondents clustered":         f"{len(clean_df):,}",
        "K-Prototypes clusters (K)":     str(kproto_result["k_selected"]),
        "LPA profiles (K)":              str(lpa_result["fit_indices"]["selected"]["K"]),
        "Global silhouette coefficient": f"{_sil:.4f}",
        "Cross-method ARI":              f"{_ari:.4f}" if _ari is not None else "N/A",
        "Ambiguous LPA respondents":     f"{lpa_result['ambiguous_count']}",
        "Outliers flagged":              f"{int(validation_result['outlier_flags'].sum())}",
    },
    artifacts=[os.path.relpath(f, _P2) for f in written] + [
        "reflection_logs/phase2_success_report.txt"
    ],
    notes=(
        "K-Prototypes: Cao initialization, elbow + silhouette K selection (Huang, 1998). "
        "LPA: BIC model selection, entropy-based ambiguity (Spurk et al., 2020; Nylund et al., 2007). "
        "Psychometrician: Gower-distance silhouette, ARI cross-validation (Rousseeuw, 1987; Steinley, 2004)."
    ),
)
written.append(_p)

print("Phase 2 outputs saved:")
for f in written:
    print(f"  {f}")

---

# Phase 3: Ground in Organizational Reality

## The Science

Clusters are statistical abstractions. A cluster centroid that reads "Trust_Leaders z = -0.71" tells you something is going on, but not *what*. To make personas useful for leadership, you need to connect these numbers to organizational context.

This phase uses two mechanisms:

### RAG (Retrieval-Augmented Generation)

The RAG agent builds a searchable knowledge base from organizational documents -- policy memos, restructuring announcements, benefits updates, team charters, FAQs. When we ask "what does low trust in leadership look like at Meridian?", the agent retrieves actual policy passages rather than making something up.

**Implementation:** TF-IDF vectorization with cosine similarity retrieval. This is deliberately simple (avoids PyTorch/embedding model dependencies). A production system would use dense embeddings for better semantic matching, but TF-IDF is sufficient for this tutorial and keeps the setup lightweight.

For background on RAG, LLMs, and embeddings, see `resources/ai_foundations/key_concepts.md`.

### Emergence Agent (Cross-Sectional Mode)

The I-O codebook defines 12 validated constructs. But what if the data contains meaningful patterns that fall *outside* those 12? The Emergence Agent scans cluster profiles for unusual construct combinations and classifies each candidate as:

- **NEW**: A genuinely novel theme not captured by the codebook
- **VARIANT**: A known construct appearing in an unusual combination
- **NOISE**: A statistical artifact, not a meaningful pattern

This operationalizes the grounded theory principle (Glaser & Strauss, 2017) that the codebook catches what we *expect* to find, and emergence detection surfaces what we did *not* expect.

In [ ]:
# Build the knowledge base from organizational documents
print("Building knowledge base from organizational documents...")
kb = build_knowledge_base(os.path.join("..", "synthetic_data", "org_documents"))
print(f"Loaded {kb['n_documents']} documents, created {kb['n_chunks']} searchable chunks.")

# Demonstrate retrieval
print("\n--- Retrieval Demonstration ---")
print("Querying the knowledge base to show what RAG finds:\n")

demo_queries = [
    "employee trust in leadership during reorganization",
    "remote work policy and flexibility",
    "career development and growth opportunities",
]
for q in demo_queries:
    results = query_knowledge_base(kb, q, top_k=2)
    print(f"Query: '{q}'")
    for r in results:
        print(f"  [{r['source']}] score={r['score']:.3f}")
        print(f"  {r['text'][:120]}...")
    print()

In [ ]:
# Ground codebook constructs in organizational documents
print("Grounding codebook constructs against organizational documents...")
grounding_result = ground_constructs(kb)
audit_trail.extend(grounding_result["audit_entries"])

print(f"\n--- Agent Reasoning ---")
print(grounding_result["reasoning"])

# Show what was found
print("\n--- Construct-to-Policy Mappings ---")
mappings = grounding_result["mappings"]
if isinstance(mappings, dict):
    for construct, data in mappings.items():
        if isinstance(data, dict) and "passages" in data:
            n_p = len(data["passages"])
            print(f"  {construct}: {n_p} relevant passage(s) found")
        else:
            print(f"  {construct}: grounding data available")

In [ ]:
# Detect emergent themes
print("Scanning for patterns outside the codebook...")
emergence_result = detect_emergent_themes(
    cluster_profiles=kproto_result["profiles"],
    codebook_constructs=list(config.NUMERIC_COLS),
)
audit_trail.extend(emergence_result["audit_entries"])

print(f"\nCandidate emergent themes: {len(emergence_result['candidates'])}")
print(f"\n--- Agent Reasoning ---")
print(emergence_result["reasoning"])

# Show theme classifications
if emergence_result["theme_report"]:
    print("\n--- Theme Classifications ---")
    report = emergence_result["theme_report"]
    if isinstance(report, list):
        for item in report:
            if isinstance(item, dict):
                print(f"  Cluster {item.get('cluster', '?')}: "
                      f"{item.get('classification', '?')} -- "
                      f"{item.get('reasoning', '')[:200]}")
    else:
        print(json.dumps(report, indent=2, default=str)[:600])

In [ ]:
# ── Phase 3 rich output ──────────────────────────────────────────────────
display(HTML(render_phase3_report(kb, grounding_result, emergence_result,
                                  run_id=RUN_ID)))

---

## Gate 3: Your Decision -- Emergent Themes

Review the patterns the Emergence Agent identified. For each candidate, consider:

- **NEW**: Is this genuinely something the codebook does not cover? If so, should it be added for future survey waves?
- **VARIANT**: Is this an interesting combination of known constructs, or just noise in the data?
- **NOISE**: Does the dismissal make sense statistically?

### Ethics Checkpoint
- Could any emergent theme actually reflect a demographic stereotype rather than a real attitudinal pattern?
- Are the organizational documents themselves biased? (e.g., do they reflect leadership's perspective more than employees'?)
- Is the codebook missing constructs that matter for underrepresented groups?

In [ ]:
print("GATE 3: EMERGENT THEME REVIEW")
print("-" * 40)
gate3_decision = input("Do you accept the theme classifications? (yes / no / investigate): ")
gate3_notes = input("Any notes? (or Enter to skip): ")

add_entry(audit_trail, "Gate 3", "Human", "Emergent theme review", {
    "decision": gate3_decision,
    "notes": gate3_notes,
    "n_candidates": len(emergence_result["candidates"]),
})
print(f"\nDecision recorded: {gate3_decision}")

In [ ]:
# ── Phase 3 output paths ─────────────────────────────────────────────────
_P3 = os.path.join("..", "outputs", "phase3_emergent_themes")
_P3_AUDIT = os.path.join(_P3, "audit_reports")
_P3_LOGS  = os.path.join(_P3, "reflection_logs")
for _d in (_P3, _P3_AUDIT, _P3_LOGS):
    os.makedirs(_d, exist_ok=True)

# ── Main report ───────────────────────────────────────────────────────────
p3_report = build_report("Phase 3: Emergent Theme Report", [
    {"heading": "RAG Grounding", "body": grounding_result["reasoning"]},
    {"heading": "Emergent Theme Analysis", "body": emergence_result["reasoning"]},
    {"heading": "Gate 3 Decision", "body": f"{gate3_decision}. {gate3_notes}"},
])

written = save_phase_outputs(
    _P3,
    report_md=p3_report,
    raw_json={
        "grounding":  {"reasoning": grounding_result["reasoning"]},
        "emergence":  {"candidates": emergence_result["candidates"],
                       "reasoning": emergence_result["reasoning"]},
    },
)

# ── Construct grounding (RAG output) ──────────────────────────────────────
_p = save_json(grounding_result["mappings"], os.path.join(_P3, "construct_grounding.json"))
written.append(_p)

# ── Emergent themes ───────────────────────────────────────────────────────
_p = save_json({
    "candidates":   emergence_result["candidates"],
    "theme_report": emergence_result["theme_report"],
}, os.path.join(_P3, "emergent_themes.json"))
written.append(_p)

# ── Knowledge base index ──────────────────────────────────────────────────
_kb_sources = sorted({c["source"] for c in kb["chunks"]}) if kb["chunks"] else []
_p = save_json({
    "n_documents": kb["n_documents"],
    "n_chunks":    kb["n_chunks"],
    "sources":     _kb_sources,
}, os.path.join(_P3, "knowledge_base_index.json"))
written.append(_p)

# ── RAG retrieval detail ──────────────────────────────────────────────────
_detail_lines = ["# RAG Agent: Retrieval Detail", "",
                 "Verbatim passages retrieved per construct with LLM relevance assessments.", ""]
_mappings = grounding_result.get("mappings", {})
for construct, data in _mappings.items():
    _detail_lines += [f"## {construct}", ""]
    if isinstance(data, dict) and data.get("passages"):
        for i, p in enumerate(data["passages"], 1):
            _detail_lines += [
                f"**Passage {i}** | Source: `{p['source']}` | Score: {p['score']:.4f}",
                "", f"> {p['text'][:400]}{'...' if len(p['text']) > 400 else ''}", "",
            ]
        if data.get("llm_assessment"):
            _detail_lines += ["**LLM Relevance Assessment:**", "", data["llm_assessment"], ""]
    else:
        _detail_lines += ["*No relevant passages found in the document corpus.*", ""]

_p = os.path.join(_P3_AUDIT, "rag_retrieval_detail.md")
with open(_p, "w", encoding="utf-8") as _f:
    _f.write("\n".join(_detail_lines))
written.append(_p)

# ── Audit trail ───────────────────────────────────────────────────────────
_combined_audit = grounding_result["audit_entries"] + emergence_result["audit_entries"]
_p = save_json(_combined_audit, os.path.join(_P3_AUDIT, "audit_trail.json"))
written.append(_p)

# ── Success report ────────────────────────────────────────────────────────
_n_grounded = sum(
    1 for v in _mappings.values()
    if isinstance(v, dict) and v.get("passages")
)
_n_cand = len(emergence_result["candidates"])
_p = write_success_report(
    os.path.join(_P3_LOGS, "phase3_success_report.txt"),
    phase="Phase 3 -- Ground in Organizational Reality",
    agents=["RAG", "Emergence"],
    status=f"COMPLETE -- Gate 3: {gate3_decision}",
    metrics={
        "Documents indexed":             str(kb["n_documents"]),
        "Text chunks":                   str(kb["n_chunks"]),
        "Constructs grounded":           f"{_n_grounded} of {len(_mappings)}",
        "Emergence candidates found":    str(_n_cand),
        "Theme classifications":         str(len(emergence_result["theme_report"]))
                                         if isinstance(emergence_result["theme_report"], list) else "0",
    },
    artifacts=[os.path.relpath(f, _P3) for f in written] + [
        "reflection_logs/phase3_success_report.txt"
    ],
    notes=(
        "RAG: TF-IDF + cosine similarity retrieval (Lewis et al., 2020). "
        "Emergence: NEW/VARIANT/NOISE classification (Braun & Clarke, 2006; Glaser & Strauss, 2017)."
    ),
)
written.append(_p)

print("Phase 3 outputs saved:")
for f in written:
    print(f"  {f}")

---

# Phase 4: Write and Validate Personas

## The Science

The Narrator Agent takes everything from the previous phases -- cluster profiles, psychological fingerprints, policy context, emergent themes -- and synthesizes them into persona narratives.

This is the step where an LLM becomes essential. No algorithm can write a narrative that captures "who these people are" from a centroid vector. But the LLM's role is tightly constrained:

### Epistemic Risk Mitigation (Nguyen & Welch, 2025)

The Narrator follows a strict protocol to prevent three common failure modes:

1. **Anthropomorphic interpretation**: The LLM cannot infer emotions or motivations beyond what the scores show. "Trust_Leaders z = -0.71" means low trust, not "feeling betrayed" or "resentful."
2. **Fabricated quotations**: No invented quotes. Every claim must trace to centroid values, fingerprints, or RAG-retrieved passages.
3. **The Oracle Effect**: The narrative must state its uncertainty explicitly rather than masking it with confident language.

Each persona includes:
- A neutral, non-stigmatizing name
- A statistical fingerprint (the numbers behind the narrative)
- Policy-experience mismatch flags (where stated policy contradicts the cluster's lived experience)
- An epistemic note disclosing evidence basis and limitations

For background on how LLMs generate text and why these guardrails matter, see `resources/ai_foundations/ai_models_explained.md`.

In [ ]:
# Generate persona narratives
print("Generating persona narratives...")
narrator_result = generate_personas(
    cluster_profiles=kproto_result["profiles"],
    construct_scores=lpa_result.get("fingerprints"),
    policy_context=grounding_result.get("mappings"),
    codebook=list(config.NUMERIC_COLS),
)
audit_trail.extend(narrator_result["audit_entries"])
personas = narrator_result["personas"]

print(f"\n--- Agent Reasoning ---")
print(narrator_result["reasoning"])

In [ ]:
# Display the personas
display(Markdown("## Draft Persona Narratives\n"))

if isinstance(personas, list):
    for i, persona in enumerate(personas):
        if isinstance(persona, dict):
            name = persona.get("persona_name", persona.get("name", f"Persona {i+1}"))
            display(Markdown(f"---\n### {name}"))

            # Statistical fingerprint
            fp = persona.get("statistical_fingerprint", {})
            if isinstance(fp, dict):
                fp_parts = []
                for dim, info in fp.items():
                    if isinstance(info, dict):
                        fp_parts.append(f"{dim}: {info.get('direction', '?')} (z={info.get('value', '?')})")
                if fp_parts:
                    display(Markdown(f"**Fingerprint:** {' | '.join(fp_parts)}"))

            # Narrative
            if "narrative" in persona:
                display(Markdown(persona["narrative"]))

            # Policy mismatches
            mismatches = persona.get("policy_mismatches", [])
            if mismatches:
                display(Markdown("**Policy-Experience Mismatches:**"))
                for m in (mismatches if isinstance(mismatches, list) else [mismatches]):
                    display(Markdown(f"- {m}"))

            # Epistemic note
            if "epistemic_note" in persona:
                display(Markdown(f"\n*{persona['epistemic_note']}*"))
        else:
            print(persona)
else:
    print(json.dumps(personas, indent=2, default=str)[:2000])

In [ ]:
# ── Phase 4 rich output: synthesis overview ──────────────────────────────
display(HTML(render_phase4_overview(
    kproto_result, lpa_result, validation_result,
    narrator_result, personas, run_id=RUN_ID,
)))

# Individual persona cards
if isinstance(personas, list):
    for _persona in personas:
        if isinstance(_persona, dict):
            display(HTML(render_persona_card(
                _persona,
                sil_per_cluster=validation_result["silhouette_per_cluster"],
                run_id=RUN_ID,
            )))

---

## Gate 4: Your Decision -- Persona Approval

Read each persona narrative carefully. For each one, ask:

- Does the narrative match the statistical fingerprint? If it says "high engagement" but the z-score is 0.08 (moderate), the narrative is overstating.
- Is the language evidence-based, or does it slip into interpretation? Watch for emotional language not supported by the data.
- Would you present this persona to leadership? Would an employee be comfortable if they recognized their group?

### Ethics Checkpoint
- Do any personas reinforce stereotypes? (e.g., "The Disengaged Veterans" as a label for long-tenure employees)
- Could these personas be misused? (e.g., to justify layoffs targeting a specific segment)
- Are they transparent about their AI-assisted construction?
- Would the people described by this persona feel fairly represented?

In [ ]:
print("GATE 4: PERSONA APPROVAL")
print("-" * 40)
gate4_decision = input("Do you approve the personas for presentation? (yes / revise / reject): ")
gate4_notes = input("Any notes? (or Enter to skip): ")

add_entry(audit_trail, "Gate 4", "Human", "Persona approval", {
    "decision": gate4_decision,
    "notes": gate4_notes,
    "n_personas": len(personas) if isinstance(personas, list) else 0,
})
print(f"\nDecision recorded: {gate4_decision}")

In [ ]:
# ── Phase 4 output paths ─────────────────────────────────────────────────
_P4 = os.path.join("..", "outputs", "phase4_persona_narratives")
_P4_AUDIT = os.path.join(_P4, "audit_reports")
_P4_LOGS  = os.path.join(_P4, "reflection_logs")
for _d in (_P4, _P4_AUDIT, _P4_LOGS):
    os.makedirs(_d, exist_ok=True)

# ── Main report ───────────────────────────────────────────────────────────
p4_report_sections = [
    {"heading": "Narrator Reasoning", "body": narrator_result["reasoning"]},
]
if isinstance(personas, list):
    for i, p in enumerate(personas):
        if isinstance(p, dict):
            name = p.get("persona_name", f"Persona {i+1}")
            body = p.get("narrative", "No narrative generated.")
            if p.get("epistemic_note"):
                body += f"\n\n*{p['epistemic_note']}*"
            p4_report_sections.append({"heading": name, "body": body})
p4_report_sections.append({"heading": "Gate 4 Decision", "body": f"{gate4_decision}. {gate4_notes}"})
p4_report = build_report("Phase 4: Draft Persona Narratives", p4_report_sections)

summary_rows = []
if isinstance(personas, list):
    for p in personas:
        if isinstance(p, dict):
            summary_rows.append({
                "persona_name":        p.get("persona_name", ""),
                "cluster_id":          p.get("cluster_id", ""),
                "size":                p.get("size", ""),
                "pct":                 p.get("pct", ""),
                "n_policy_mismatches": len(p.get("policy_mismatches", [])),
            })
persona_summary_df = pd.DataFrame(summary_rows)

written = save_phase_outputs(
    _P4,
    report_md=p4_report,
    dataframes={"persona_summary": persona_summary_df} if not persona_summary_df.empty else None,
    raw_json={"personas": personas, "reasoning": narrator_result["reasoning"]},
)

# ── Personas JSON (clean, no envelope) ───────────────────────────────────
_p = save_json(personas, os.path.join(_P4, "personas.json"))
written.append(_p)

# ── Personas markdown (Gate 4 deliverable) ────────────────────────────────
_personas_md_lines = [
    "# Phase 4: Workforce Personas -- Gate 4 Deliverable", "",
    "**Agents:** Narrator  |  **Protocol:** Epistemic Risk Mitigation (Nguyen & Welch, 2025)", "",
    "> Every claim in the narratives below traces to a statistical centroid value.",
    "> The I-O psychologist retains final interpretive authority over all personas.", "",
    "---", "",
]
if isinstance(personas, list):
    for p in personas:
        if isinstance(p, dict):
            name = p.get("persona_name", "Unnamed Persona")
            _personas_md_lines += [f"## {name}", ""]
            fp = p.get("statistical_fingerprint", {})
            if isinstance(fp, dict):
                _personas_md_lines += ["| Construct | Direction | Z-score |", "|-----------|-----------|---------|"]
                for dim, info in fp.items():
                    if isinstance(info, dict):
                        _personas_md_lines.append(
                            f"| {dim} | {info.get('direction','?')} | {info.get('value','?')} |"
                        )
                _personas_md_lines.append("")
            if p.get("narrative"):
                _personas_md_lines += [p["narrative"], ""]
            mismatches = p.get("policy_mismatches", [])
            if mismatches:
                _personas_md_lines += ["**Policy-Experience Mismatches:**"]
                for m in (mismatches if isinstance(mismatches, list) else [mismatches]):
                    _personas_md_lines.append(f"- {m}")
                _personas_md_lines.append("")
            if p.get("epistemic_note"):
                _personas_md_lines += [f"*{p['epistemic_note']}*", ""]
            _personas_md_lines.append("---")
            _personas_md_lines.append("")

_p = os.path.join(_P4, "personas.md")
with open(_p, "w", encoding="utf-8") as _f:
    _f.write("\n".join(_personas_md_lines))
written.append(_p)

# ── Ethics checkpoint ─────────────────────────────────────────────────────
_ethics_md = "\n".join([
    "# Ethics Checkpoint: Gate 4 Persona Review",
    "",
    "Complete this checklist before approving personas for leadership presentation.",
    "",
    "## 1. Input Bias",
    "- [ ] Careless-responding removal was demographically proportionate (review Phase 1 bias audit)",
    "- [ ] No demographic group was disproportionately screened out",
    "",
    "## 2. Clustering Bias",
    "- [ ] No cluster maps suspiciously onto a single demographic group without theoretical justification",
    "- [ ] Outlier respondents (Phase 2) are acknowledged, not silently excluded",
    "",
    "## 3. Narrative Bias",
    "- [ ] All narrative claims trace to centroid values in the statistical fingerprint",
    "- [ ] No emotional language is used beyond what z-scores support",
    "- [ ] No persona name stigmatises or stereotypes a group",
    "",
    "## 4. Retrieval Bias",
    "- [ ] Org documents reflect employee perspectives, not only leadership communications",
    "- [ ] Policy-experience mismatches are flagged, not smoothed over",
    "",
    "## 5. Epistemic Risk (Nguyen & Welch, 2025)",
    "- [ ] Every persona includes an epistemic note disclosing AI assistance",
    "- [ ] Uncertainty is stated explicitly (no Oracle Effect language)",
    "- [ ] No fabricated quotes or unverifiable claims",
    "",
    "## 6. Intended Use",
    "- [ ] Personas will not be used to make individual employment decisions",
    "- [ ] Leadership has been briefed on the probabilistic nature of cluster assignments",
    "- [ ] A human I-O psychologist has reviewed and approved each narrative",
    "",
    "## Acknowledgement",
    "",
    f"Gate 4 decision: **{gate4_decision}**",
    "",
    f"Notes: {gate4_notes or '(none)'}",
    "",
    "*This checkpoint must be completed and signed before sharing personas with leadership.*",
])
_p = os.path.join(_P4_AUDIT, "ethics_checkpoint.md")
with open(_p, "w", encoding="utf-8") as _f:
    _f.write(_ethics_md)
written.append(_p)

# ── Audit trail ───────────────────────────────────────────────────────────
_p = save_json(narrator_result["audit_entries"], os.path.join(_P4_AUDIT, "audit_trail.json"))
written.append(_p)

# ── Success report ────────────────────────────────────────────────────────
_n_p = len(personas) if isinstance(personas, list) else 0
_has_policy = grounding_result.get("mappings") is not None
_p = write_success_report(
    os.path.join(_P4_LOGS, "phase4_success_report.txt"),
    phase="Phase 4 -- Write and Validate Personas",
    agents="Narrator",
    status=f"PENDING GATE 4 -- decision: {gate4_decision}",
    metrics={
        "Personas generated":            str(_n_p),
        "Policy context (RAG grounding)":"attached" if _has_policy else "not attached",
        "LPA fingerprints attached":     "yes" if lpa_result.get("fingerprints") else "no",
        "Ethics checkpoint":             "completed (see audit_reports/ethics_checkpoint.md)",
        "Epistemic protocol":            "Nguyen & Welch (2025)",
    },
    artifacts=[os.path.relpath(f, _P4) for f in written] + [
        "reflection_logs/phase4_success_report.txt"
    ],
    notes=(
        "Epistemic risk mitigation: no fabricated quotes, no inferences beyond z-scores, "
        "statistical anchoring for every claim (Nguyen & Welch, 2025). "
        "I-O psychologist retains final interpretive authority."
    ),
)
written.append(_p)

print("Phase 4 outputs saved:")
for f in written:
    print(f"  {f}")

---

# Bonus: Longitudinal Mode

## The Science

When a follow-up survey arrives, two additional agents activate:

### Continuity Agent

Maps each follow-up respondent to the nearest baseline cluster centroid using composite distance (weighted Euclidean on numerics + Hamming on categoricals). Critical detail: the follow-up data is standardized using **baseline parameters** (mean and SD from the first survey), not the follow-up's own statistics. This prevents masking genuine attitude shifts -- if morale dropped across the board, you want to see that, not have it normalized away.

Respondents whose distance to every centroid exceeds the threshold (0.35) are flagged as **weak-fit** -- they no longer belong to any baseline group.

### Emergence Agent (Longitudinal Mode)

Takes the weak-fit pool and runs a K+1 test: can these "misfits" be split into 2 or 3 coherent sub-groups? If K=2 produces a silhouette > 0.50, there may be a genuinely new workforce segment that did not exist in the baseline.

### Transition Matrix

Shows how respondents moved between clusters from baseline to follow-up. This is the longitudinal story: which segments grew, shrank, or dissolved.

In [ ]:
# Clean follow-up data through the same Data Steward
print("Screening follow-up survey data...")
steward_fu = run_data_steward(followup_df)
clean_fu = steward_fu["clean_df"]
audit_trail.extend(steward_fu["audit_entries"])
print(f"Follow-up: {len(clean_fu):,} respondents after screening")

# Align to baseline centroids
print("\nAligning follow-up respondents to baseline clusters...")
continuity_result = align_to_baseline(
    followup_df=clean_fu,
    baseline_centroids=kproto_result["centroids"],
    baseline_labels=kproto_result["labels"],
    baseline_df=clean_df,
)
audit_trail.extend(continuity_result["audit_entries"])

print(f"\nWeak-fit respondents: {continuity_result['weak_fit_count']} "
      f"({continuity_result['weak_fit_count']/len(clean_fu)*100:.1f}%)")
print("These respondents no longer fit into any baseline cluster.")

print(f"\n--- Agent Reasoning ---")
print(continuity_result["reasoning"])

if continuity_result["transition_matrix"] is not None:
    print("\n--- Transition Matrix ---")
    print("Rows = baseline cluster, Columns = follow-up cluster")
    print("Values show proportion of each baseline cluster that moved to each follow-up cluster.")
    display(continuity_result["transition_matrix"])

In [ ]:
# K+1 test on weak-fit respondents
weak_fit_df = clean_fu[continuity_result["weak_fit_mask"]]

if len(weak_fit_df) >= 30:
    print(f"Running K+1 test on {len(weak_fit_df)} weak-fit respondents...")
    new_seg_result = test_new_segments(weak_fit_df)
    audit_trail.extend(new_seg_result["audit_entries"])

    print("\nResults by K:")
    for k, res in new_seg_result["k_plus_1_results"].items():
        if isinstance(res, dict) and "bic" in res:
            sil_str = f", silhouette={res['silhouette']:.3f}" if res.get("silhouette") else ""
            print(f"  K={k}: BIC={res['bic']:.1f}{sil_str}, sizes={res['sizes']}")

    if new_seg_result["new_segment_candidates"]:
        print("\nNew segment candidates detected!")
        for c in new_seg_result["new_segment_candidates"]:
            print(f"  {c}")
    else:
        print("\nNo strong evidence for new segments in the weak-fit pool.")
else:
    print(f"Only {len(weak_fit_df)} weak-fit respondents (minimum 30 for K+1 test).")
    print("Skipping K+1 analysis.")

---

# Audit Trail and Final Outputs

Every agent action and every gate decision has been logged. The audit trail below is the complete record of this analysis -- who did what, when, and why. This is essential for reproducibility and for demonstrating that the I-O psychologist maintained interpretive authority throughout.

In [ ]:
# Generate and display the audit report
audit_report = generate_audit_report(audit_trail)

print("=" * 70)
print("CUMULATIVE AUDIT TRAIL")
print("=" * 70)
print(audit_report["report_text"])
print(f"\nTotal entries: {audit_report['n_entries']}")
print(f"Phases: {', '.join(audit_report['phases'])}")
print(f"Agents: {', '.join(audit_report['agents'])}")

In [ ]:
# Save audit trail in both formats
save_audit_csv(audit_trail, os.path.join("..", "outputs", "audit_trail", "audit_trail.csv"))
save_output(audit_trail, os.path.join("..", "outputs", "audit_trail", "audit_trail.json"))

print("Audit trail saved:")
print("  outputs/audit_trail/audit_trail.csv  (open in Excel to review)")
print("  outputs/audit_trail/audit_trail.json  (raw for reproducibility)")

---

## What You Have Now

Check the `outputs/` folder. You should see:

| Folder | Contents | Who It Is For |
|--------|----------|---------------|
| `phase1_data_quality_report/` | report.md + screening_results.csv + results.json | You and your team -- verify the screening was appropriate |
| `phase2_cluster_validation/` | report.md + cluster_assignments.csv + results.json | You and your team -- review every respondent's cluster assignment |
| `phase3_emergent_themes/` | report.md + results.json | You -- decide what to add to the codebook |
| `phase4_persona_narratives/` | report.md + persona_summary.csv + results.json | Leadership -- the personas themselves |
| `audit_trail/` | audit_trail.csv + audit_trail.json | Governance -- full record of every decision |

The `.md` reports can be shared directly. The `.csv` files open in Excel. The `.json` files allow exact reproduction.

### Next Steps

- Run this on your own organization's data (replace the CSVs in `synthetic_data/`)
- Expand the codebook with domain-specific constructs (edit `resources/io_codebook.md`)
- Add organizational documents to `synthetic_data/org_documents/` for richer RAG grounding

### References

- Braun, V. & Clarke, V. (2006). Using thematic analysis in psychology. *Qualitative Research in Psychology*, 3(2), 77-101.
- Curran, P. G. (2016). Methods for the detection of carelessly invalid responses in survey data. *JESP*, 66, 4-19.
- Glaser, B. G. & Strauss, A. L. (2017). *The Discovery of Grounded Theory*. Routledge.
- Hallgren, K. A. (2012). Computing inter-rater reliability. *Tutorials in Quantitative Methods for Psychology*, 8(1).
- Huang, Z. (1998). Extensions to the k-means algorithm for clustering large data sets with categorical values. *Data Mining and Knowledge Discovery*, 2(3), 283-304.
- Nguyen, D. C. & Welch, C. (2025). Generative AI in qualitative data analysis. *Organizational Research Methods*.
- Papp, Baker, Dutcher, & McClelland (2026). The Survey Data Quality Evaluation Model. *Teaching of Psychology*.
- Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to cluster validation. *J. Computational and Applied Mathematics*, 20.
- Spurk, D. et al. (2020). Latent profile analysis: A review and "how to" guide. *Journal of Vocational Behavior*, 120.

In [ ]:
print("Pipeline complete.")
print(f"\nAudit trail: {audit_report['n_entries']} entries")
print(f"Phases covered: {', '.join(audit_report['phases'])}")
print(f"Agents used: {', '.join(audit_report['agents'])}")
print(f"Model: {config.MODEL}")
print(f"\nAll outputs in: outputs/")